In [1]:
import sqlite3
import pandas as pd


In [2]:
conn = sqlite3.connect('/Users/nameerakhan/Desktop/Miscellaneous/bnpl-credit-pred/data/db/homecredit.db')

In [3]:
cursor = conn.cursor()


In [4]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

In [5]:
print("Tables in database:")
for table in tables:
    print(f"-{table[0]}")

Tables in database:
-application_train
-bureau
-bureau_balance
-previous_application
-pos_cash_balance
-installments_payments
-credit_card_balance


In [6]:
if tables: #schema 
    table_name = tables[0][0]
    print(f"\nSchema of {table_name}':")
    cursor.execute(f"PRAGMA table_info({table_name})")
    columns = cursor.fetchall()
    for col in columns:
        print(f"  {col[1]}: {col[2]}")


Schema of application_train':
  SK_ID_CURR: INTEGER
  TARGET: INTEGER
  NAME_CONTRACT_TYPE: TEXT
  CODE_GENDER: TEXT
  FLAG_OWN_CAR: TEXT
  FLAG_OWN_REALTY: TEXT
  CNT_CHILDREN: INTEGER
  AMT_INCOME_TOTAL: REAL
  AMT_CREDIT: REAL
  AMT_ANNUITY: REAL
  AMT_GOODS_PRICE: REAL
  NAME_TYPE_SUITE: TEXT
  NAME_INCOME_TYPE: TEXT
  NAME_EDUCATION_TYPE: TEXT
  NAME_FAMILY_STATUS: TEXT
  NAME_HOUSING_TYPE: TEXT
  REGION_POPULATION_RELATIVE: REAL
  DAYS_BIRTH: INTEGER
  DAYS_EMPLOYED: INTEGER
  DAYS_REGISTRATION: REAL
  DAYS_ID_PUBLISH: INTEGER
  OWN_CAR_AGE: REAL
  FLAG_MOBIL: INTEGER
  FLAG_EMP_PHONE: INTEGER
  FLAG_WORK_PHONE: INTEGER
  FLAG_CONT_MOBILE: INTEGER
  FLAG_PHONE: INTEGER
  FLAG_EMAIL: INTEGER
  OCCUPATION_TYPE: TEXT
  CNT_FAM_MEMBERS: REAL
  REGION_RATING_CLIENT: INTEGER
  REGION_RATING_CLIENT_W_CITY: INTEGER
  WEEKDAY_APPR_PROCESS_START: TEXT
  HOUR_APPR_PROCESS_START: INTEGER
  REG_REGION_NOT_LIVE_REGION: INTEGER
  REG_REGION_NOT_WORK_REGION: INTEGER
  LIVE_REGION_NOT_WORK_REGIO

In [13]:
app = pd.read_sql("""
SELECT 
    SK_ID_CURR, TARGET,
    AMT_CREDIT, AMT_ANNUITY, AMT_INCOME_TOTAL, AMT_GOODS_PRICE,
    AMT_CREDIT/NULLIF(AMT_INCOME_TOTAL, 0) AS credit_income_ratio,
    AMT_ANNUITY/NULLIF(AMT_GOODS_PRICE, 0) AS annuity_income_ratio,
    AMT_CREDIT/NULLIF(AMT_GOODS_PRICE, 0) AS credit_goods_ratio,
    DAYS_BIRTH / -365.0 AS age_years,
    DAYS_EMPLOYED / -365.0 AS years_employed,
    CASE WHEN DAYS_EMPLOYED > 0 THEN 1 ELSE 0 END AS is_unemployed,
    REGION_POPULATION_RELATIVE, REGION_RATING_CLIENT, 
    EXT_SOURCE_1, EXT_SOURCE_2, EXT_SOURCE_3,
    (COALESCE(EXT_SOURCE_1,0) + COALESCE(EXT_SOURCE_2,0) + COALESCE(EXT_SOURCE_3,0)) / 3.0 AS ext_source_mean, 
    (CASE WHEN EXT_SOURCE_1 IS NULL THEN 1 ELSE 0 END + 
    CASE WHEN EXT_SOURCE_2 IS NULL THEN 1 ELSE 0 END +
    CASE WHEN EXT_SOURCE_3 IS NULL THEN 1 ELSE 0 END) AS ext_source_nulls,
    CASE NAME_CONTRACT_TYPE WHEN 'Cash loans' THEN 1 ELSE 0 END AS is_cash_loan, 
    CASE CODE_GENDER        WHEN 'M'          THEN 1 ELSE 0 END AS is_male,
    CASE NAME_EDUCATION_TYPE
                WHEN 'Higher education' THEN 3
                WHEN 'Incomplete higher' THEN 2
                WHEN 'Secondary / secondary special' THEN 1
                ELSE 0 END AS education_encoded,
    FLAG_DOCUMENT_3, FLAG_DOCUMENT_6, FLAG_MOBIL, FLAG_WORK_PHONE
FROM application_train
    


    """,conn)

In [14]:
bureau_agg = pd.read_sql("""
SELECT SK_ID_CURR,
    COUNT(*) AS bureau_loan_count,
    SUM(CASE WHEN CREDIT_ACTIVE='Active' THEN 1 ELSE 0 END) AS bureau_active_loans,
    SUM(AMT_CREDIT_SUM) AS bureau_total_credit,
    SUM(AMT_CREDIT_SUM_DEBT) AS bureau_total_debt,
    SUM(CASE WHEN AMT_CREDIT_SUM_OVERDUE > 0 THEN 1 ELSE 0 END) AS bureau_overdue_count
FROM bureau GROUP BY SK_ID_CURR 
    """,conn)

In [15]:
prev_agg = pd.read_sql("""
        SELECT SK_ID_CURR,
            COUNT(*) AS prev_app_count,
            SUM(CASE WHEN NAME_CONTRACT_STATUS='Approved'  THEN 1 ELSE 0 END) AS prev_approved,
            SUM(CASE WHEN NAME_CONTRACT_STATUS='Refused'   THEN 1 ELSE 0 END) AS prev_refused,
            SUM(CASE WHEN NAME_CONTRACT_STATUS='Approved'  THEN 1 ELSE 0 END) * 1.0 /
                NULLIF(COUNT(*), 0) AS prev_approval_rate
        FROM previous_application GROUP BY SK_ID_CURR
    """, conn)

In [16]:
install_agg = pd.read_sql("""
        SELECT SK_ID_CURR,
            COUNT(*) AS install_count,
            SUM(CASE WHEN DAYS_ENTRY_PAYMENT > DAYS_INSTALMENT THEN 1 ELSE 0 END) AS install_late_count,
            AVG(DAYS_INSTALMENT - DAYS_ENTRY_PAYMENT) AS install_avg_delay_days
        FROM installments_payments GROUP BY SK_ID_CURR
    """, conn)

In [17]:
conn.close()

In [18]:
df = app
df = df.merge(bureau_agg, on ="SK_ID_CURR", how="left")
df = df.merge(prev_agg, on="SK_ID_CURR",how="left")
df = df.merge(install_agg, on="SK_ID_CURR", how="left")


In [19]:
df

,SK_ID_CURR,TARGET,AMT_CREDIT,AMT_ANNUITY,AMT_INCOME_TOTAL,AMT_GOODS_PRICE,credit_income_ratio,annuity_income_ratio,credit_goods_ratio,age_years,...,bureau_total_credit,bureau_total_debt,bureau_overdue_count,prev_app_count,prev_approved,prev_refused,prev_approval_rate,install_count,install_late_count,install_avg_delay_days
0,100002,1,406597.5,24700.5,202500.0,351000.0,2.007889,0.070372,1.158397,25.920548,...,865055.565,245781.00,0.0,1.0,1.0,0.0,1.000000,19.0,0.0,20.421053
1,100003,0,1293502.5,35698.5,270000.0,1129500.0,4.790750,0.031606,1.145199,45.931507,...,1017400.500,0.00,0.0,3.0,3.0,0.0,1.000000,25.0,0.0,7.160000
2,100004,0,135000.0,6750.0,67500.0,135000.0,2.000000,0.050000,1.000000,52.180822,...,189037.800,0.00,0.0,1.0,1.0,0.0,1.000000,3.0,0.0,7.666667
3,100006,0,312682.5,29686.5,135000.0,297000.0,2.316167,0.099955,1.052803,52.068493,...,NaN,NaN,NaN,9.0,5.0,1.0,0.555556,16.0,0.0,19.375000
4,100007,0,513000.0,21865.5,121500.0,513000.0,4.222222,0.042623,1.000000,54.608219,...,146250.000,0.00,0.0,6.0,6.0,0.0,1.000000,66.0,16.0,3.636364
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307506,456251,0,254700.0,27558.0,157500.0,225000.0,1.617143,0.122480,1.132000,25.553425,...,NaN,NaN,NaN,1.0,1.0,0.0,1.000000,7.0,0.0,36.285714
307507,456252,0,269550.0,12001.5,72000.0,225000.0,3.743750,0.053340,1.198000,56.917808,...,NaN,NaN,NaN,1.0,1.0,0.0,1.000000,6.0,1.0,2.833333
307508,456253,0,677664.0,29979.0,153000.0,585000.0,4.429176,0.051246,1.158400,41.002740,...,3960000.000,1795833.00,0.0,2.0,2.0,0.0,1.000000,14.0,1.0,14.500000
307509,456254,1,370107.0,20205.0,171000.0,319500.0,2.164368,0.063239,1.158394,32.769863,...,45000.000,0.00,0.0,2.0,2.0,0.0,1.000000,19.0,0.0,19.000000


In [20]:
agg_cols = [c for c in df.columns if c.startswith(("bureau_","prev_","install_"))]
df[agg_cols] = df[agg_cols].fillna(0)


In [21]:
df

,SK_ID_CURR,TARGET,AMT_CREDIT,AMT_ANNUITY,AMT_INCOME_TOTAL,AMT_GOODS_PRICE,credit_income_ratio,annuity_income_ratio,credit_goods_ratio,age_years,...,bureau_total_credit,bureau_total_debt,bureau_overdue_count,prev_app_count,prev_approved,prev_refused,prev_approval_rate,install_count,install_late_count,install_avg_delay_days
0,100002,1,406597.5,24700.5,202500.0,351000.0,2.007889,0.070372,1.158397,25.920548,...,865055.565,245781.00,0.0,1.0,1.0,0.0,1.000000,19.0,0.0,20.421053
1,100003,0,1293502.5,35698.5,270000.0,1129500.0,4.790750,0.031606,1.145199,45.931507,...,1017400.500,0.00,0.0,3.0,3.0,0.0,1.000000,25.0,0.0,7.160000
2,100004,0,135000.0,6750.0,67500.0,135000.0,2.000000,0.050000,1.000000,52.180822,...,189037.800,0.00,0.0,1.0,1.0,0.0,1.000000,3.0,0.0,7.666667
3,100006,0,312682.5,29686.5,135000.0,297000.0,2.316167,0.099955,1.052803,52.068493,...,0.000,0.00,0.0,9.0,5.0,1.0,0.555556,16.0,0.0,19.375000
4,100007,0,513000.0,21865.5,121500.0,513000.0,4.222222,0.042623,1.000000,54.608219,...,146250.000,0.00,0.0,6.0,6.0,0.0,1.000000,66.0,16.0,3.636364
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307506,456251,0,254700.0,27558.0,157500.0,225000.0,1.617143,0.122480,1.132000,25.553425,...,0.000,0.00,0.0,1.0,1.0,0.0,1.000000,7.0,0.0,36.285714
307507,456252,0,269550.0,12001.5,72000.0,225000.0,3.743750,0.053340,1.198000,56.917808,...,0.000,0.00,0.0,1.0,1.0,0.0,1.000000,6.0,1.0,2.833333
307508,456253,0,677664.0,29979.0,153000.0,585000.0,4.429176,0.051246,1.158400,41.002740,...,3960000.000,1795833.00,0.0,2.0,2.0,0.0,1.000000,14.0,1.0,14.500000
307509,456254,1,370107.0,20205.0,171000.0,319500.0,2.164368,0.063239,1.158394,32.769863,...,45000.000,0.00,0.0,2.0,2.0,0.0,1.000000,19.0,0.0,19.000000
